In [11]:
import os
import random
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend

In [12]:
def pad(data: bytes, block_size: int = 16) -> bytes:
    n = block_size - (len(data) % block_size)
    return data + bytes([n] * n)

def unpad(data: bytes) -> bytes:
    n = data[-1]             # last byte tells you how many padding bytes
    return data[:-n]

### Testing

In [13]:
print(pad(b"Hello"))
print(pad(b"HelloHelloHello1"))

b'Hello\x0b\x0b\x0b\x0b\x0b\x0b\x0b\x0b\x0b\x0b\x0b'
b'HelloHelloHello1\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10\x10'


### AES CBC Manual (Learning Purpose)

In [25]:
def aes_ecb_encrypt(key: bytes, block: bytes) -> bytes:
    cipher = Cipher(algorithms.AES(key=key), mode=modes.ECB(), backend=default_backend())
    enc = cipher.encryptor()
    return enc.update(block) + enc.finalize()

def aes_ecb_decrypt_block(key: bytes, block: bytes) -> bytes:
    cipher = Cipher(algorithms.AES(key=key), mode=modes.ECB(), backend=default_backend())
    dec = cipher.decryptor()
    return dec.update(block) + dec.finalize()

def xor_bytes(a: bytes, b: bytes) -> bytes:
    result = bytes(x ^ y for x,y in zip(a,b))
    return result

def aes_cbc_encrypt(key: bytes, iv: bytes, plaintext: bytes) -> bytes:
    plaintext = pad(plaintext)
    ciphertext = b""
    prev_block = iv

    for i in range(0, len(plaintext), 16):
        block = plaintext[i: i + 16]
        xored = xor_bytes(block, prev_block)
        encrypted = aes_ecb_encrypt(key, xored)
        ciphertext += encrypted
        prev_block = encrypted
    return ciphertext

def aes_cbc_decrypt(key: bytes, iv: bytes, ciphertext: bytes) -> bytes:
    plaintext = b""
    prev_block = iv

    for i in range(0, len(ciphertext), 16):
        block = ciphertext[i: i + 16]
        decrypted = aes_ecb_decrypt_block(key, block)
        xored = xor_bytes(decrypted, prev_block)
        plaintext += xored
        prev_block = block
    return unpad(plaintext)

### Testing

In [28]:
key = os.urandom(16)
iv = os.urandom(16)
plaintext = b"Hello World, this is a testing code section"

print(f"Key      : {key.hex()}")
print(f"IV       : {iv.hex()}")
print(f"Plaintext: {plaintext}\n")

ciphertext = aes_cbc_encrypt(key, iv, plaintext)

decrypted = aes_cbc_decrypt(key, iv, ciphertext)

print(f"[ENCRYPTED]: {ciphertext.hex()}")
print(f"[DECRYPTED]: {decrypted}")

Key      : d6548be284ea3dea6059be554d139b1b
IV       : 03c5bc8009c0524b2c3304fb84e5fd34
Plaintext: b'Hello World, this is a testing code section'

[ENCRYPTED]: 4d4839de1fad53a1e6cf2a51685606b933ebbc77bb4432b9a315df45ddb0211932f0a2b1401c481ea71dee12fd3b5a35
[DECRYPTED]: b'Hello World, this is a testing code section'
